# Sprint 1 Live — Understand and extend an AI service

An **application** guides a person's interaction; a **service** exposes behavior that another program can call. In these four workshops, use Colab as the client and inspect a small FastAPI service to explain how its route, data contract and optional model call fit together. The supplied example names identify runnable code; no business project or Campus ZIP is required.

## Outcomes and navigation

| Session | Learning outcome | Start here |
|---|---|---|
| LS01 | Distinguish an AI application from an AI service; tour the provided FastAPI scaffold and identify routes, inputs, outputs, and model call. | [Predict, call, locate](#scrollTo=markdown-02) |
| LS02 | Extend the provided scaffold with a new endpoint, define its input/output contract, and version the route. | [Extend the scaffold](#scrollTo=markdown-05) |
| LS03 | Define clear input/output contracts for service endpoints and apply /v1, /v2 versioning to model and prompt changes. | [Compare versions](#scrollTo=markdown-08) |
| LS04 | Demo a new endpoint added to the scaffold with a clean contract and a versioned route. | [Demonstrate and review](#scrollTo=markdown-11) |

### Start a working copy

1. Save an editable copy of this notebook in Colab. Connect a standard Python CPU runtime; no GPU is needed. Run [setup](#scrollTo=setup) and wait for the printed workspace path before moving on.
2. Run the current session's cells in order using each cell's play button. Cell labels such as `ls02-start` match the live-session guide and the comment at the top of each code cell. **Do not use Run all:** the notebook deliberately pauses for your source edits and model choice.
3. Add your predictions and explanations in your own text cells. Save the notebook **and** download the [Live source ZIP](#scrollTo=markdown-14) after every session. Saving the notebook does not save runtime files; see the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

Setup installs the course lock from a fresh clone of `main`. It creates `PROJECT`, your editable practice folder, or reuses an existing marked Live folder without replacing its files. `REPO` is the downloaded reference source: **edit `PROJECT`, not `REPO`**. Start cells launch a notebook-owned service and replace only its previous process. Its `127.0.0.1` address belongs to this runtime; use the notebook HTTP cells to call it. Opening that address in your laptop browser will not reach Colab.

**Expected after setup:** a workspace path, source revision and two non-secret model choices. Setup does not start a service, build an index or call a provider. Both provider switches later in the notebook stay `False` unless you deliberately opt in.

### Resume safely

| Situation | What to do |
|---|---|
| Same runtime, same work | Keep `LIVE_ARCHIVE` empty and the same `LIVE_FOLDER`; run setup, then the current session's start cell. Source edits remain. Setup stops the old service. |
| Fresh runtime, saved work | Upload your own Live ZIP through Files. Set `LIVE_ARCHIVE` to its full runtime path, choose a `LIVE_FOLDER` that does not exist, and fill both `RESTORED_MODELS` from the model-choice record saved with your notebook. Run setup once. |
| Restore succeeded | Review the restored source before starting it. Clear `LIVE_ARCHIVE` before another setup run in this runtime. Re-enter credentials and prepare the index only if opting into generation. |
| Existing unmarked folder or restore destination | Choose a new folder name. Do not delete a learner folder to get past the error. Campus exports are a different format. |

If clone/install fails, keep your work and inspect the first error; retry setup only after the network/package issue is resolved. If Colab requests a runtime restart, save and download current work first, then resume as above. The [Live handoff guide](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/live-handoff.md) describes the separate archive format.


In [ ]:
# Cell: setup
from pathlib import Path
import os, sys, subprocess, tempfile, json, secrets, time
if globals().get("service") is not None:
    service.stop()
service = None
REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path(tempfile.mkdtemp(prefix="module-b-live-source-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
previous_helper_path = globals().get("LIVE_HELPER_PATH")
import importlib
if previous_helper_path in sys.path:
    sys.path.remove(previous_helper_path)
LIVE_HELPER_PATH = str(REPO / "src")
sys.path.insert(0, LIVE_HELPER_PATH)
# Select the new checkout even when setup is rerun in the same runtime.
for module_name in tuple(sys.modules):
    if module_name == "module_b" or module_name.startswith("module_b."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import httpx
from module_b.live import prepare_live_workspace, export_live_workspace, restore_live_workspace
from module_b.runtime import ServiceProcess
# On a fresh runtime, upload your own Live ZIP using the Files panel.
LIVE_ARCHIVE = ""  # Example: "/content/sprint-1-live-123.zip"; keep empty for new practice.
LIVE_FOLDER = Path.cwd() / "fieldcare-sprint-1-live"
# When restoring, explicitly select the same approved models used in that work.
RESTORED_MODELS = {"OPENROUTER_MODEL": "", "OPENROUTER_MODEL_V2": ""}
from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
if LIVE_ARCHIVE:
    if any(model not in CHAT_MODELS for model in RESTORED_MODELS.values()):
        raise ValueError("Fill both RESTORED_MODELS with your saved approved model choices before restoring.")
    PROJECT = restore_live_workspace(LIVE_ARCHIVE, LIVE_FOLDER)
else:
    PROJECT = prepare_live_workspace(REPO, LIVE_FOLDER)
CALLERS = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
caller_keys = {name: secrets.token_urlsafe(32) for name in CALLERS}
previous_models = {name: globals().get("service_env", {}).get(name) for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
service_env = {variable: caller_keys[name] for name, variable in CALLERS.items()}
service_env["FIELDCARE_WORK_DIR"] = str((PROJECT / "var").resolve())
for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2"):
    service_env[name] = (RESTORED_MODELS[name] if LIVE_ARCHIVE else
                         previous_models[name] or os.environ.get(name, DEFAULT_CHAT_MODEL))
    if service_env[name] not in CHAT_MODELS:
        raise ValueError(f"Select an approved {name}.")
print("Configured models:", {name: service_env[name] for name in RESTORED_MODELS})
supported = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
incomplete = {"question": "Which filter checks are documented?"}
print("Live workspace:", PROJECT)
print("Source main:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
print("No provider or index is needed for health, validation and clarification.")

from module_b.live import start_live_service


## LS01 — Predict, call, locate

An **endpoint** is a callable operation selected by an HTTP method and path. A **schema** declares the data's fields, types and constraints. JSON carries those fields between the client and service. An HTTP status describes request handling; the body's `status` describes the example's result state.

**Guided trace.** Run `ls01-source`. In `main.py`, find router registration; in `routes.py`, find `POST /v1/diagnose`; in `schemas.py`, locate `DiagnosticRequest` and `DiagnosticResponse`. The remaining files show the early return, resource selection, retrieval and `client.chat` call. Follow those names in that order rather than trying to memorize every line.

Before running `code-03`, predict which steps each supplied request reaches. Run `ls01-start` first; its printed URL means the owned service passed its startup check. Then call the service. No provider key is needed.


In [ ]:
# Cell: ls01-source
for name in ("main.py", "routes.py", "schemas.py", "service.py", "resources.py", "orchestration.py", "model.py"):
    print(f"\n--- service/fieldcare/{name} ---")
    print((PROJECT / "service/fieldcare" / name).read_text())


In [ ]:
# Cell: ls01-start
service = start_live_service(PROJECT, "main", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
# Cell: code-03
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    health = client.get("/health")
    print("health", health.status_code, health.json())
    for label, body in [("missing optional context", incomplete), ("invalid input", {})]:
        response = client.post("/v1/diagnose", json=body)
        print(label, response.status_code, response.json())

**Compare with your prediction.** The health request returns HTTP `200`; its preparation fields can still say the provider/index is unavailable. A question without `equipment_id` returns `200` and `needs_clarification`; `{}` returns `422` with validation details. Read `status_code` and the body separately. `mode: live` appears on clarification too, so it does not establish that a model ran.

Trace `clarification_for` before `graph_for`, then the conditional retrieval-to-generation path and `client.chat`. Input validation can reject a request before the handler; the handler can return clarification before any provider work. An application would use these distinctions to ask for missing context or display a validation message.

**Your independent check — 14 minutes.** Choose an incomplete or invalid example, write your prediction, then change one input and run it in a new code cell using the demonstrated HTTP pattern. Record the method/path, input schema, output schema, possible model-call location, actual status/body and a justified client action. Mark the reached and skipped steps. A copied source listing alone does not complete the task.

If a call reports connection refused, rerun `ls01-start` and retry the same request. A `422` is an observed validation response, not a notebook execution error or model refusal. Keep source unchanged in this session.


### Optional provider preparation — after the deterministic checks

You can complete the core source/contract exercises without a provider. Opt in only to observe generated behavior: set `PREPARE_PROVIDER = True` and run the next cell. It stops the owned service, requests the provider key privately, and runs `python -m fieldcare.prepare_index`. Embedding the supplied synthetic documents uses provider quota; keep the key out of code cells, outputs and archives.

After preparation succeeds, set `RUN_PROVIDER = True` in `ls01-provider` to restart and submit the supported example. Expect an actual response or an explicit error: `200`/`ready` with answer and citations is different from `200`/`needs_clarification`, `503` for missing preparation, or `502` for a provider failure. Exact generated wording varies. Inspect a cited document against one answer claim; a healthy process does not establish answer quality.

If preparation fails, leave generation pending and return to deterministic checks by rerunning the relevant start cell. Reset the switches to `False` after the deliberate provider run to avoid accidental repeat calls. Restored ZIPs contain neither credentials nor the index.


In [ ]:
# Cell: prepare-provider
PREPARE_PROVIDER = False
if PREPARE_PROVIDER:
    if service is not None:
        service.stop()
    from module_b.openrouter import require_openrouter_key
    require_openrouter_key(prompt=True)
    preparation_env = {**os.environ, **service_env,
                       "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
    subprocess.run([sys.executable, "-m", "fieldcare.prepare_index"],
                   cwd=PROJECT, env=preparation_env, check=True)
else:
    print("Provider preparation pending; deterministic checks remain available.")

In [ ]:
# Cell: ls01-provider
RUN_PROVIDER = False
if RUN_PROVIDER:
    service = start_live_service(PROJECT, "main", service_env, service)
    with httpx.Client(base_url=service.base_url, timeout=120) as client:
        response = client.post("/v1/diagnose", json=supported)
        print(response.status_code, response.json())
else:
    print("Generated-answer evidence pending.")

## LS02 — Extend the scaffold

A scaffold supplies reusable structure. Extending it means connecting a new operation to that structure: declare its contract, bind its behavior and register its router. A **router** groups handlers; `include_router` makes those handlers reachable in the running application. A prompt changes instructions passed to a model; it does not replace schema validation.

**Guided example.** Run `ls02-source`. In `brief_routes.py`, locate the named input/output models, early clarification, `BRIEF_PROMPT` and `graph_for(system_prompt=...)`. In `live_extension.py`, find registration. Run `ls02-start`, then `code-06` with the supplied brief path. Expect clarification `200` and invalid input `422` on both the brief and original route. The cell also displays the contract registered in OpenAPI, the service's machine-readable API description; no browser tunnel is needed.

### Edit source in Colab

1. Open Files and expand the folder printed as `PROJECT`, then `service/fieldcare`. Open `brief_routes.py` and `live_extension.py` in the text editor. The printed source in a cell output is read-only.
2. For a new file, use the folder's New file action and give it the required `.py` name. If your Files view does not offer creation, create that file in a local text editor and upload it into this exact folder. Refresh Files after uploading. Do not paste a completed implementation into notebook setup.
3. Save the file in the editor, then rerun `ls02-source` to confirm saved content for the supplied files. For a new file, add a scratch cell that reads its path with `.read_text()`. An editor buffer is not the source the service has loaded.
4. Rerun `ls02-start` after every saved source change. It starts a new process from the edited files. Then rerun the same HTTP check; setup is not the reload step.

### Your independent extension — 14 minutes

Add a second output format to this example: a checklist of documented checks followed by one sentence identifying what still needs verification. The concept to demonstrate is a new callable contract with a separately bound prompt.

- Create `checklist_schemas.py` with explicitly named request/response classes inheriting `DiagnosticRequest`/`DiagnosticResponse`. Preserve the trimmed `1–2000` character question, optional bounded equipment value, four response fields and clarification states.
- Create `checklist_routes.py` using the brief operation as a pattern. Bind your classes and your own `CHECKLIST_PROMPT` to `POST /v1/diagnose-checklist`. Preserve shared safety instructions, early clarification and provider-error handling.
- Import and include your router once in `live_extension.py`. Keep the original and brief operations unchanged. Save and restart, set `ROUTE_TO_REVIEW` to your new path, then rerun `code-06`.

Write a contract card in your own text cell: method/path; required and optional fields; constraints; output fields and states; meaningful errors. Have a peer use it without a verbal explanation. Completion requires your own saved declarations, prompt binding, registration and actual new/original-route results. Running only the supplied brief example does not finish this task; the notebook contains no completed checklist implementation.


In [ ]:
# Cell: ls02-source
print((PROJECT / "service/fieldcare/brief_routes.py").read_text())
print((PROJECT / "service/fieldcare/live_extension.py").read_text())

In [ ]:
# Cell: ls02-start
service = start_live_service(PROJECT, "live_extension", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
# Cell: code-06
ROUTE_TO_REVIEW = "/v1/diagnose-brief"  # After authoring, choose "/v1/diagnose-checklist".
RUN_PROVIDER = False
with httpx.Client(base_url=service.base_url, timeout=120) as client:
    api = client.get("/openapi.json").json()
    operation = api["paths"].get(ROUTE_TO_REVIEW, {}).get("post")
    if operation is None:
        raise ValueError("Route not registered: check its method/path, save the registration and rerun ls02-start.")
    print("Registered input:", operation.get("requestBody"))
    print("Registered output:", operation["responses"]["200"])
    for path in [ROUTE_TO_REVIEW, "/v1/diagnose"]:
        for label, body in [("clarification", incomplete), ("invalid", {})]:
            response = client.post(path, json=body)
            print(path, label, response.status_code, response.json())
            if label == "clarification":
                assert response.status_code == 200 and response.json().get("status") == "needs_clarification", "Compare the preserved optional input and clarification branch."
            else:
                assert response.status_code == 422, "Compare the required question declaration."
        if RUN_PROVIDER:
            response = client.post(path, json=supported)
            print(path, "provider comparison", response.status_code, response.json())
print("Demo checked; your extension is still required." if ROUTE_TO_REVIEW == "/v1/diagnose-brief"
      else "Calls checked. Now inspect your named schemas, prompt binding and source changes with a peer.")


**Check what the evidence supports.** Compare the registered input/output names and each response with your contract card. These calls test validation, clarification and preservation of the original operation; they do not exercise your prompt's generated wording. Source inspection establishes which prompt is configured. A separately opted-in provider call is needed to observe its effect, and one answer does not establish general quality.

| Symptom | Check and recovery |
|---|---|
| Startup/import error | Read the named file and line; check the saved filename, import, indentation and router name. Repair that source, then restart. |
| New path missing or `404` | Check the exact method/path and `include_router` in `live_extension.py`; save and restart. |
| Expected clarification becomes `422` | Compare your inherited optional field and constraints with the original schema. |
| Original route changes too | Inspect edits to shared files. Undo only your unintended change and retest both routes. |

Save the notebook and Live ZIP now if the session is ending. After export stops the service, rerun the relevant start cell before another request.


## LS03 — Compare contracts and change one version

A **contract** is the caller/service agreement about inputs and outputs. A **version** lets callers deliberately select an agreement. A new path alone does not protect an old caller if both handlers use a changed shared schema, prompt or model setting.

The worked example keeps `/v1/diagnose` at a `2000` character question limit and adds `/v2/diagnose` with `PilotRequest` capped at `600`. V1 uses `OPENROUTER_MODEL`; v2 passes `openrouter_model_v2()` explicitly to `graph_for`. Each selection is independently checked against the approved list. Your LS02 operation remains registered because `live_versions.py` imports `live_extension.py`.

1. Run `ls03-source`. Locate each version's schema, prompt and model binding. Predict what a `700` character question without equipment will do on each path.
2. Run `ls03-model-before` to capture both selected names and source digests. A digest is a fingerprint used here to detect file changes.
3. Choose a different approved v2 model and enter it as the quoted `PILOT_MODEL` in `ls03-model-change`. Its initial empty value deliberately stops execution until you choose. Run it **before editing source**: it checks that v1 and source bindings remain unchanged.
4. Run `ls03-start`, then `code-09`. Expect the old path to accept the long input and clarify, while v2 rejects it. Short incomplete input clarifies on both; an empty body fails validation on both. Inspect the actual results before concluding that the expected boundary held.

Selecting a model and checking source does not call that model. Keep generation claims separate. If you rerun the model experiment, first restore the intended comparison state and run `ls03-model-before` again; do not overwrite a useful before/after record without saving it.


In [ ]:
# Cell: ls03-source
print((PROJECT / "service/fieldcare/pilot_schemas.py").read_text())
print((PROJECT / "service/fieldcare/pilot_routes.py").read_text())
print((PROJECT / "service/fieldcare/live_versions.py").read_text())

In [ ]:
# Cell: ls03-model-before
import hashlib
# These are non-secret configuration values and source digests, not provider observations.
binding_paths = sorted((PROJECT / "service/fieldcare").glob("*.py"))
source_before_model_change = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in binding_paths}
models_before = {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
print("Before:", models_before)
print("Approved choices:", sorted(CHAT_MODELS))


In [ ]:
# Cell: ls03-model-change
PILOT_MODEL = ""  # Choose a different approved model from the printed list.
if PILOT_MODEL not in CHAT_MODELS or PILOT_MODEL == models_before["OPENROUTER_MODEL_V2"]:
    raise ValueError("Choose another course-approved pilot model before continuing.")
assert service_env["OPENROUTER_MODEL"] == models_before["OPENROUTER_MODEL"]
assert source_before_model_change == {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in binding_paths}
service_env["OPENROUTER_MODEL_V2"] = PILOT_MODEL
print("After:", {name: service_env[name] for name in models_before})
selection_env = {**os.environ, **service_env,
                 "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
subprocess.run([sys.executable, "-m", "clients.version_models"], cwd=PROJECT, env=selection_env, check=True)
print("V1 selection, prompts and schemas stayed unchanged. Restart both routes for the contract checks.")


In [ ]:
# Cell: ls03-start
service = start_live_service(PROJECT, "live_versions", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
# Cell: code-09
OLD_PATH, NEW_PATH = "/v1/diagnose", "/v2/diagnose"
long_request = {"question": "filter " + "x" * 693}
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    for path in [OLD_PATH, NEW_PATH]:
        for label, body in [("short", incomplete), ("700 characters", long_request), ("invalid", {})]:
            response = client.post(path, json=body)
            print(path, label, response.status_code, response.json().get("status"))
# Verify the LS02 registration survived the later composition.
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    response = client.post("/v1/diagnose-checklist", json=incomplete)
    print("Your retained LS02 operation:", response.status_code, response.json())
    if response.status_code == 404:
        print("LS02 work is incomplete: create and register your own operation before review.")


**Your independent version change — 14 minutes.** Keep the model-only before/after record first. Then edit only the formatting instruction appended to `PILOT_PROMPT` in `pilot_routes.py`; choose your own changed format while preserving the shared safety instructions. Save this as a separate change, restart `ls03-start` and repeat `code-09`. Do not change schemas, v1 prompt/model or your LS02 operation.

Record which source/configuration changed, which original-caller agreement stayed fixed, and what the calls actually tested. If provider preparation is ready, use a separate HTTP comparison with `supported` on both paths to observe generated behavior. Otherwise state that the formatting effect remains unobserved; clarification does not use the prompt.

Write a migration condition: identify an existing caller, the input/output adaptation it needs, and an observation that would justify retiring its old route. **Deprecation** is notice to migrate, not removal: `deprecated=True` marks API documentation but does not redirect or delete the route. Keep v1 callable while callers still depend on it.

If the retained checklist reports `404`, return to your LS02 source and registration, then restart `ls03-start`. Do not replace the task with the supplied pilot. If both paths reject the long request, inspect whether a shared schema was unintentionally changed. Keep the model experiment and prompt edit separate so you can explain each effect.


## LS04 — Demonstrate and review your extension

Demonstrate the endpoint you added in Colab, explain its named input/output contract and versioned path, and use a request to test that explanation. This is a formative concept demonstration; the supplied brief/pilot examples do not substitute for your own extension.

**Entry check:** your own `checklist_schemas.py`, `checklist_routes.py` and registration must exist. In a fresh runtime restore your saved ZIP, run `ls03-source` and `ls03-start`, then `code-09` to recreate the comparison request. Keep the model experiment's saved record; repeating it is unnecessary for the demonstration.

Run `code-12` only after that entry check. Expect the new operation to clarify for accepted incomplete input, reject invalid input and preserve the original route's longer request. The cell stops if the required source/registered path is missing or the response disagrees. Passing these checks does not verify the prompt's generated behavior.

Take turns as owner, caller reviewer and source reviewer. Each owner gets four minutes to show source and an exchange, two for a peer's changed-input prediction, one to locate a mismatch and one to record it. In a pair, use the third turn for reciprocal follow-up checks; a solo learner reviews with the instructor. Every learner explains their own work.


In [ ]:
# Cell: code-12
REVIEW_PATH = "/v1/diagnose-checklist"  # Your LS02 extension, retained in live_versions.
required_files = [PROJECT / "service/fieldcare" / name
                  for name in ("checklist_schemas.py", "checklist_routes.py")]
if not all(path.is_file() for path in required_files):
    raise ValueError("Complete or restore your own LS02 source before demonstrating it.")
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    api = client.get("/openapi.json").json()
    if REVIEW_PATH not in api["paths"]:
        raise ValueError("Your operation is not registered in this running app. Check source, then rerun ls03-start.")
    for path, body, expected in [(REVIEW_PATH, incomplete, 200), (REVIEW_PATH, {}, 422),
                                 ("/v1/diagnose", long_request, 200)]:
        response = client.post(path, json=body)
        print(path, response.status_code, response.json())
        assert response.status_code == expected, "Compare this result with your contract before continuing."
        if expected == 200:
            assert response.json().get("status") == "needs_clarification", "Explain the returned state; a 200 alone is insufficient."
print("Selected contract checks passed. Explain your declarations and one limitation to your reviewer.")


**Independent verification — 12 minutes.** Choose one mismatch or missing check raised by the review. Save the starting observation, predict the result that would resolve it, then make one justified repair or test the existing behavior. Restart only if source/configuration changed. Repeat the request and relevant original-route check. A justified no-change decision needs an observation too.

In your own notebook text cell, record your method/path and schema names, observed request/response, the code explaining it, your peer's challenge, the repair or confirming check, and one remaining limitation. Keep the LS03 old/new contract and separate model/prompt comparisons alongside it. A replay must be labelled as a replay; a blocked run needs its exact pending check.

## Summary

You can now connect a callable route to its data contract and selected behavior, explain why some requests never reach a model, and test what existing callers retain when another version changes. Save that source and explanation for the next concept workshop.


## Save your Live practice

Run `save-checkpoint` after any session, not only at the end of the sprint. It stops the owned service, saves the two non-secret model selections and exports your current Python source plus selected `.md`/`.json` evidence. It does not grade completion. An unfinished attempt is still worth saving.

1. Save your notebook text cells and outputs. The ZIP does **not** contain this notebook; your predictions, contract card and reviewer notes remain in the saved notebook unless you separately write a safe `.md`/`.json` file under `PROJECT/evidence/`.
2. Run the export cell, keep the printed model choices with your notebook, and download the exact ZIP path it prints through the Files panel. Keep it on durable storage before disconnecting.
3. In a later fresh runtime, upload that ZIP and use the restore instructions at the top. Archives exclude `.env`, credentials, the document index and raw logs. Never paste keys into an evidence file.

To continue working immediately after saving, rerun your current session's start cell before making another request. Saving/exporting does not replace or update a separate Campus project.


In [ ]:
# Cell: save-checkpoint
if service is not None:
    service.stop()
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/live-models.json").write_text(json.dumps(
    {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}, indent=2) + "\n")
print("Save these configured model choices with your notebook:",
      {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")})
CHECKPOINT_OUT = PROJECT.parent / f"sprint-1-live-{time.time_ns()}.zip"
print("Download:", export_live_workspace(PROJECT, CHECKPOINT_OUT))